# TF Causal Model — Metabolic-GT PR Evaluation

Same setup as `tf_causal_zscores.ipynb` through `df_tf`, then evaluates the
z-score signal against ground-truth TF labels built from iML1515 + the
Ledezma-Tejeida 2022 sensor annotations.

Three GT definitions per (perturbation, TF):
1. **substrate** — effector is a direct substrate of the perturbed enzyme
2. **substrate ∪ product** — effector is a substrate or product
3. **downstream@k** — effector is reachable within `k` reaction steps from
   the products (currency-metabolite-filtered BFS)

PR curves use `|Zscore_emp|` as the score, and are computed only on cells
where the perturbation is in iML1515 *and* the TF has at least one
BiGG-resolved effector (the `coverage` mask).

In [ ]:
import sys

sys.path.insert(0, "/workspace/src")

from pathlib import Path

import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import plotnine as gg

import tf_prediction as tfp
from essential.data import load_regulondb_full

In [ ]:
# ADATA_PATH = "/workspace/data/de122_lce75/adata_de122_lce75_merged.h5ad"
# EXPERIMENT_SUBSET = "lce75"
# PERT_COL = "target"
# CTRL_KEY = "nontargeting"
# MIN_LIB = 1e3

ADATA_PATH = "/workspace/data/260309_lce75_genomescale_ezrdm_glu_preprocessed/260309_lce75_genomescale_ezrdm_glu_preprocessed.h5ad"
PERT_COL = "top_target_unthresholded"
CTRL_KEY = "nontargeting"
MIN_LIB = 1e3

N_EPOCHS = 100
BATCH_SIZE = 512
LR = 1e-3
KEY = jax.random.PRNGKey(0)

MODEL_JSON = "/workspace/data/05142026_metabolome/iML1515.json"
SENSOR_CSV = (
    "/workspace/experiments/06222026_tf/tf_effector_reference/"
    "ledezmatejeida_2022_tf_annotations_resolved.csv"
)
DOWNSTREAM_DEPTH = 2

## 1. Data

In [ ]:
# adata1 = sc.read_h5ad(ADATA_PATH)
# adata2 = sc.read_h5ad(
#     "/workspace/data/Nov2025_DE122_genomescale_EZRDM_Glu_newpipeline_preprocessed/Nov2025_DE122_genomescale_EZRDM_Glu_newpipeline_preprocessed.h5ad"
# )
# adata = adata1.concat(
#     adata2, batch_key="experiment", batch_categories=["lce75", "de122"]
# )

In [ ]:
adata = sc.read_h5ad(ADATA_PATH)
adata.obs["_lib"] = np.asarray(adata.layers["reads"].sum(1)).ravel()
adata = adata[
    (adata.obs["_lib"] > MIN_LIB)
    # & (adata.obs["experiment"] == EXPERIMENT_SUBSET)
    & adata.obs[PERT_COL].notna()
].copy()
adata.var_names = adata.var_names.str.lower()
adata.obs[PERT_COL] = adata.obs[PERT_COL].str.lower()

tfp.prepare_layers(adata)
adata.X = adata.layers["counts"]
print(f"{adata.n_obs:,} cells x {adata.n_vars:,} genes")

In [ ]:
ref_db = load_regulondb_full()
ref_db = ref_db[ref_db["ri_type"].str.startswith("TF")].copy()

tf_info = (
    ref_db.assign(TF=ref_db["regulator_gene"].str.lower())
    .groupby("TF")
    .agg(
        n_edges=("target_gene", "size"),
        frac_uncertain=("confidenceLevel", lambda s: (s == "W").mean()),
        frac_nan=("confidenceLevel", lambda s: s.isna().mean()),
    )
    .reset_index()
)

valid_tfs = set(tf_info.query("n_edges >= 2")["TF"])
ref_db["regulator_gene"] = ref_db["regulator_gene"].str.lower()
ref_db["target_gene"] = ref_db["target_gene"].str.lower()
ref_db = ref_db[ref_db["regulator_gene"].isin(valid_tfs)].copy()

all_tfs = set(ref_db["regulator_gene"].unique())
perts = adata.obs[PERT_COL]
tf_perts = (set(perts.unique()) - {CTRL_KEY}) & all_tfs
non_tf_perts = (set(perts.unique()) - {CTRL_KEY}) - all_tfs

adata_train = adata[perts.isin(tf_perts) | (perts == CTRL_KEY)].copy()
adata_val = adata[perts.isin(non_tf_perts)].copy()
print(
    f"Train: {adata_train.n_obs:,} cells  ({len(tf_perts)} TF perturbations + controls)"
)
print(f"Val:   {adata_val.n_obs:,} cells  ({len(non_tf_perts)} non-TF perturbations)")

In [ ]:
mask = tfp.build_tf_mask(adata.var_names, ref_db)
n_genes, n_tfs = adata.n_vars, len(mask["tf_genes"])
print(
    f"{n_tfs} TF features  |  {int(mask['Amask_tf'].sum()):,} edges  |  "
    f"{int((mask['Amask_tf'].sum(1) > 0).sum())} genes with >=1 regulator"
)

ctrl_mask = np.asarray(adata_train.obs[PERT_COL] == CTRL_KEY)
std = tfp.TFStandardizer.fit(
    adata_train, tf_cols=mask["tf_cols"], control_mask=ctrl_mask
)
arrays_train = std.transform(adata_train)
arrays_val = std.transform(adata_val)
arrays_all = std.transform(adata)

## 2. Train (causal RegulonDB-masked NB)

In [ ]:
# model = tfp.TFCellBox/(
model = tfp.TFRegression(
    n_genes=n_genes,
    n_tfs=n_tfs,
    x_mean=jnp.asarray(std.ctrl_lcp_mean),
    Amask_tf=jnp.asarray(mask["Amask_tf"]),
)

print("Training...")
state, history = tfp.fit(
    model,
    # *arrays_train,
    *arrays_all,
    n_epochs=100,
    batch_size=BATCH_SIZE,
    lr=LR,
    key=KEY,
)
print(f"Final train NLL: {history['train_nll'][-1]:.4f}")

plt.plot(history["train_nll"])
plt.xlabel("epoch")
plt.ylabel("train NLL / cell")
plt.title("training curve")

In [ ]:
pert_labels = np.asarray(adata_train.obs[PERT_COL])
mom = tfp.per_perturbation_moments(
    model,
    state.params,
    *arrays_train,
    pert_labels,
    batch_size=BATCH_SIZE,
)
mom = tfp.mask_perturbed_gene(mom, adata.var_names)  # drop each pert's own gene

W_eff = np.asarray(state.params["W"]) * mask["Amask_tf"]
z = tfp.tf_zscores(mom, W_signed=np.sign(W_eff), Amask_tf=mask["Amask_tf"])

In [ ]:
v = np.ravel(mom.Rnum / (1e-6 + mom.Vsum))
v_ = v[~np.isnan(v)]

bins = np.linspace(-3, 3, 100)
plt.hist(v_, bins=bins)

In [ ]:
# for i in range(200):
#     plt.hist(z.Z_emp[:, i], bins=200)
#     plt.show()

In [ ]:
# n_targets per TF = column sums of the regulon mask (genes regulated by each TF)
n_targets = np.asarray(mask["Amask_tf"].sum(0)).ravel()
max_abs_z = np.abs(z.Z_emp).max(axis=0)  # per-TF max over perturbations

assert n_targets.shape == max_abs_z.shape, (n_targets.shape, max_abs_z.shape)

fig, ax = plt.subplots(figsize=(6, 5))
ax.scatter(n_targets, max_abs_z, s=12, alpha=0.6)
ax.set_xlabel("n_targets (genes per TF)")
ax.set_ylabel(r"$\max_p\ |Z_{emp}|$")
ax.set_title("per-TF max |Z_emp| vs regulon size")
ax.set_xscale("log")
plt.show()

## 3. Per-(perturbation, TF) z-scores

In [ ]:
pert_labels = np.asarray(adata_val.obs[PERT_COL])
mom = tfp.per_perturbation_moments(
    model,
    state.params,
    *arrays_val,
    pert_labels,
    batch_size=BATCH_SIZE,
)
mom = tfp.mask_perturbed_gene(mom, adata.var_names)  # drop each pert's own gene

W_eff = np.asarray(state.params["W"]) * mask["Amask_tf"]
z = tfp.tf_zscores(mom, W_signed=np.sign(W_eff), Amask_tf=mask["Amask_tf"])

tf_genes = mask["tf_genes"]
df_tf = (
    tfp.tf_score_frame(z, mom.perts, tf_genes)
    .sort_values("abs_Zscore_emp", ascending=False)
    .reset_index(drop=True)
)
df_tf.head(20)

In [ ]:
# mom = tfp.per_perturbation_moments(
#     model,
#     state.params,
#     *arrays_val,
#     pert_labels,
#     batch_size=BATCH_SIZE,
# )
# mom_fixed = tfp.mask_perturbed_gene(mom, adata.var_names)  # drop each pert's own gene

# W_eff = np.asarray(state.params["W"]) * mask["Amask_tf"]
# z = tfp.tf_zscores(mom, W_signed=np.sign(W_eff), Amask_tf=mask["Amask_tf"])
# z_fixed = tfp.tf_zscores(mom_fixed, W_signed=np.sign(W_eff), Amask_tf=mask["Amask_tf"])

## 4. Metabolic GT PR curves

GT labels are computed on the same `(mom.perts, mask['tf_genes'])` axes as
the z-score matrix. `z.Z_emp` is shape `(n_perts, n_tfs)` and aligns row-for-
row with the GT arrays, so we can feed `np.abs(z.Z_emp)` directly to
`pr_curve`.

In [ ]:
sensor = pd.read_csv(SENSOR_CSV)
mm = tfp.MetabolicModel(MODEL_JSON)

# restrict the perturbation axis to enzymatic perts (genes present in iML1515)
enzymatic_idx = np.array([i for i, p in enumerate(mom.perts) if mm.has_enzyme(p)])
enzymatic_perts = [mom.perts[i] for i in enzymatic_idx]
print(f"enzymatic perturbations: {len(enzymatic_perts)} / {len(mom.perts)}")

gt = tfp.build_gt_labels(
    mm,
    sensor,
    perturbations=enzymatic_perts,
    tf_names=list(tf_genes),
    depth=DOWNSTREAM_DEPTH,
)
scores = np.abs(z.Z_emp)[enzymatic_idx]

print(f"axes: {gt.coverage.shape[0]} perturbations x {gt.coverage.shape[1]} TFs")
print(f"coverage cells (evaluable): {int(gt.coverage.sum()):,} / {gt.coverage.size:,}")
print(
    f"  TFs with >=1 BiGG effector: {int(gt.coverage.any(axis=0).sum())} / {gt.coverage.shape[1]}"
)
print()
print(
    f"positives - substrate            : {int((gt.Y_substrate & gt.coverage).sum()):,}"
)
print(
    f"positives - substrate or product : {int((gt.Y_substrate_or_product & gt.coverage).sum()):,}"
)
print(
    f"positives - downstream@{DOWNSTREAM_DEPTH}        : {int((gt.Y_downstream & gt.coverage).sum()):,}"
)

In [ ]:
pr_sub = tfp.pr_curve(scores, gt.Y_substrate, coverage=gt.coverage)
pr_sop = tfp.pr_curve(scores, gt.Y_substrate_or_product, coverage=gt.coverage)
pr_down = tfp.pr_curve(scores, gt.Y_downstream, coverage=gt.coverage)

for name, pr in [
    ("substrate", pr_sub),
    ("substrate or product", pr_sop),
    (f"downstream@{DOWNSTREAM_DEPTH}", pr_down),
]:
    print(
        f"{name:22s}  AUPRC={pr.auprc:.4f}  "
        f"prevalence={pr.prevalence:.4f}  "
        f"lift={pr.auprc / max(pr.prevalence, 1e-12):.2f}x  "
        f"({pr.n_positives}/{pr.n_total})"
    )

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
for name, pr, color in [
    # ("substrate", pr_sub, "C0"),
    ("substrate ∪ product", pr_sop, "C1"),
    # (f"downstream@{DOWNSTREAM_DEPTH}", pr_down, "C2"),
]:
    ax.plot(
        pr.recall,
        pr.precision,
        color=color,
        label=f"{name}  (AUPRC={pr.auprc:.3f})",
    )
    ax.axhline(pr.prevalence, color=color, ls=":", lw=0.8, alpha=0.6)
ax.set_xlabel("recall")
ax.set_xscale("log")
ax.set_ylabel("precision")
ax.set_title(f"|Z_emp| vs metabolic GT")
ax.legend(loc="best", fontsize=9)
ax.grid(True, ls="-", lw=0.4, alpha=0.4)

In [ ]:
# Precision@K curve: y = precision among the top-K highest-|Z_emp| predictions,
# x = K. Same coverage mask + finite filter as `pr_curve`, so the curves are
# directly comparable to the PR plot above.
fig, ax = plt.subplots(figsize=(6, 5))

cov = gt.coverage.ravel()
s_all = scores.ravel()[cov]
finite = np.isfinite(s_all)
s_all = s_all[finite]
order = np.argsort(-s_all, kind="stable")  # rank by descending score
k = np.arange(1, len(order) + 1)

for name, Y, color in [
    # ("substrate", gt.Y_substrate, "C0"),
    ("substrate ∪ product", gt.Y_substrate_or_product, "C1"),
    # (f"downstream@{DOWNSTREAM_DEPTH}", gt.Y_downstream, "C2"),
]:
    y_all = Y.ravel()[cov][finite].astype(bool)[order]
    precision_at_k = np.cumsum(y_all) / k
    ax.plot(
        k, precision_at_k, color=color, label=f"{name}  (prevalence={y_all.mean():.3f})"
    )
    ax.axhline(y_all.mean(), color=color, ls=":", lw=0.8, alpha=0.6)  # random baseline

ax.set_xscale("log")
ax.set_xlabel("top K discoveries (ranked by |Z_emp|)")
ax.set_ylabel("precision in top K")
ax.set_title(f"precision@K")
ax.legend(loc="best", fontsize=9)
ax.grid(True, ls="-", lw=0.4, alpha=0.4)

## 4b. Named GT table — per-row metabolite identities

`build_gt_labels` gives the boolean matrices the PR curves need. This block
attaches *readable metabolite names* to each `(perturbation, TF)` row so a
poorly-predicted regulon can be inspected directly:

- `enzyme_substrates` / `enzyme_sop` / `enzyme_downstream` — what the perturbed
  enzyme touches in iML1515. These depend on the **perturbation only**, so they
  are shown for *every* enzymatic perturbation, independent of TF coverage.
- `effector_names` — the metabolite(s) the TF senses (from the sensor CSV)
- `shared_substrate` / `shared_sop` / `shared_downstream` — the metabolite that
  actually links them (empty ⇒ no metabolic explanation)
- `coverage` — `False` where the pert isn't in iML1515 or the TF has no
  resolved effector (so `gt_*=False` there means *can't say*, not *no link*)

Two joins: enzyme sets by `perturbation`, effector overlap by `(perturbation,
TF)`. This is notebook-side enrichment (string-key joins), kept out of `src`.
The GT booleans are recomputed here as set intersections and cross-checked
against `build_gt_labels` below.

In [ ]:
import json

# effectors per TF (names + compartment-stripped BiGG ids), from the sensor CSV
resolved = sensor[sensor["resolved"]].assign(
    TF=lambda d: d["Transcription factor"].str.lower()
)
eff_by_tf = resolved.groupby("TF").agg(
    effector_names=("effector_name", lambda s: sorted(set(s))),
    effector_bigg=("bigg_id", lambda s: sorted(set(s))),
)

# bigg_id -> readable name, for the enzyme's substrate/product/downstream lists
with open(MODEL_JSON) as f:
    _mets = json.load(f)["metabolites"]
bigg_name = {}
for m in _mets:
    bigg_name.setdefault(m["id"].rsplit("_", 1)[0], m.get("name"))
names = lambda ids: ", ".join(sorted({bigg_name.get(b, b) for b in ids}))

# (a) per-PERTURBATION enzyme metabolite sets — depend on the pert only, so they
#     are shown for every enzymatic pert regardless of TF coverage.
pert_sets = {}
enz_rows = []
for p in df_tf["perturbation"].unique():
    if not mm.has_enzyme(p):
        continue  # not in iML1515
    subs = mm.enzyme_substrates(p)
    sop = subs | mm.enzyme_products(p)
    down = mm.enzyme_downstream(p, depth=DOWNSTREAM_DEPTH)
    pert_sets[p] = (subs, sop, down)
    enz_rows.append(
        {
            "perturbation": p,
            "enzyme_substrates": names(subs),
            "enzyme_sop": names(sop),
            "enzyme_downstream": names(down),
        }
    )
enzyme_info = pd.DataFrame(enz_rows)

# (b) per-(PERTURBATION, TF) effector overlap + coverage — needs both an
#     enzymatic pert and a TF with >=1 resolved effector.
tf_set = set(tf_genes)
gt_rows = []
for p, (subs, sop, down) in pert_sets.items():
    for tf, e in eff_by_tf.iterrows():
        if tf not in tf_set:
            continue
        eff = set(e["effector_bigg"])
        shared_sub, shared_sop, shared_down = eff & subs, eff & sop, eff & down
        gt_rows.append(
            {
                "perturbation": p,
                "TF": tf,
                "coverage": True,
                "effector_names": ", ".join(e["effector_names"]),
                "shared_substrate": names(shared_sub),
                "shared_sop": names(shared_sop),
                "shared_downstream": names(shared_down),
                "gt_substrate": bool(shared_sub),
                "gt_sop": bool(shared_sop),
                "gt_downstream": bool(shared_down),
            }
        )
gt_long = pd.DataFrame(gt_rows)

# cross-check the recomputed booleans against build_gt_labels (coverage cells)
assert int(gt_long["gt_substrate"].sum()) == int((gt.Y_substrate & gt.coverage).sum())
assert int(gt_long["gt_sop"].sum()) == int(
    (gt.Y_substrate_or_product & gt.coverage).sum()
)
assert int(gt_long["gt_downstream"].sum()) == int((gt.Y_downstream & gt.coverage).sum())
print(f"gt_long: {len(gt_long):,} covered (pert, TF) pairs — matches build_gt_labels")

# enzyme sets join by perturbation (shown for all enzymatic perts); effector
# overlap joins by (perturbation, TF). Unmatched coverage -> False (not "no link").
df_merged = df_tf.merge(enzyme_info, on="perturbation", how="left").merge(
    gt_long, on=["perturbation", "TF"], how="left"
)
df_merged["coverage"] = df_merged["coverage"].fillna(False)
for c in ["gt_substrate", "gt_sop", "gt_downstream"]:
    df_merged[c] = df_merged[c].fillna(False)


df_metabolism_merged = df_merged.query("perturbation in @enzymatic_perts")

In [ ]:
# # df_merged.query("abs_Zscore_emp >= 3.0").to_csv(
# #     "/workspace/experiments/06222026_tf/06242026_tf_perturbation_scores_all_v2.csv",
# #     index=False,
# # )
# df_metabolism_merged.query("abs_Zscore_emp >= 3.0").to_csv(
#     "/workspace/experiments/06222026_tf/06242026_tf_perturbation_scores_metabolism_v3.csv",
#     index=False,
# )

## 5. High-confidence hits

For each (perturbation, TF) cell with `|Zscore_emp| >= THRESHOLD` and coverage,
show its GT membership under the three definitions. Lets you read off which
predictions are corroborated by the metabolic model.

In [ ]:
df_metabolism_merged.query("abs_Zscore_emp >= 3.0")["TF"].value_counts().sort_values(
    ascending=False
).head(50)

In [ ]:
# hits = df_metabolism_merged.query("abs_Zscore_emp >= 3.0")
# hits = df_metabolism_merged.head(500)
# print(f"{len(hits)} (perturbation, TF)")
# print(f"  also GT substrate         : {int(hits['gt_substrate'].sum())}")
# print(f"  also GT substrate or prod : {int(hits['gt_sop'].sum())}")
# print(
#     f"  also GT downstream@{DOWNSTREAM_DEPTH}      : {int(hits['gt_downstream'].sum())}"
# )


hits = (
    df_metabolism_merged.groupby("TF")
    .apply(lambda x: x.sort_values("abs_Zscore_emp", ascending=False).head(1))
    .reset_index(drop=True)
    .query("abs_Zscore_emp >= 3.0")
)
valid_hit = hits.query("gt_sop")
n_characterized_tfs = len(valid_hit["TF"].unique())
print(f"including {n_characterized_tfs} characterized TFs")
display(
    f"top 1 hit per TF with known effectors (substrate ∪ product overlap) — {len(valid_hit)} hits / {len(hits)} candidate hits"
)
display_columns = {
    "perturbation": "Enzyme perturbation",
    "TF": "TF",
    "effector_names": "Know effectors associated with TF",
    "enzyme_sop": "Enzyme substrates ∪ products (extractable from metabolic model)",
    "abs_Zscore_emp": "Model score",
}
display(
    valid_hit.rename(columns=display_columns)
    .loc[:, display_columns.values()]
    .style.set_properties(**{"white-space": "normal", "text-align": "left"})
)

TOP_K = 5
hits = (
    df_metabolism_merged.groupby("TF")
    .apply(lambda x: x.sort_values("abs_Zscore_emp", ascending=False).head(TOP_K))
    .reset_index(drop=True)
    .query("abs_Zscore_emp >= 3.0")
)
valid_hit = hits.query("gt_sop")
display(
    f"top {TOP_K} hits per TF with known effectors (substrate ∪ product overlap) — {len(valid_hit)} hits / {len(hits)} candidate hits"
)
display_columns = {
    "perturbation": "Enzyme perturbation",
    "TF": "TF",
    "effector_names": "Know effectors associated with TF",
    "enzyme_sop": "Enzyme substrates ∪ products (extractable from metabolic model)",
    "abs_Zscore_emp": "Model score",
}
display(
    valid_hit.rename(columns=display_columns)
    .loc[:, display_columns.values()]
    .style.set_properties(**{"white-space": "normal", "text-align": "left"})
)

In [ ]:
# Per-TF metabolite counts over the top-K perturbations.
# Avoid parsing `enzyme_sop` (names are joined with ", " but BiGG names contain
# commas, e.g. "5,10-methylenetetrahydrofolate" -> ambiguous split). Instead use
# `pert_sets[p] = (subs, sop, down)` from cell 68d65ea3, which holds the actual
# BiGG-id SETS in memory. Group on ids (unambiguous), attach names for display.
# `hits` is the TOP_K-per-TF table from the cell above.
#
# Currency metabolites (cofactors, ions, ubiquitous byproducts) appear in almost
# every reaction's substrate/product set, so they swamp the counts without being
# informative. Compartment-stripped BiGG ids, matching `pert_sets`.
CURRENCY_METABOLITES = {
    "atp",
    "adp",
    "h",
    "h2o",
}

# groupby(dropna=False): keep TFs whose effector_names is NaN (no resolved
# effector). Without it pandas silently drops every NaN-keyed group, leaving
# only the ~75 characterized TFs.
EMPTY = (set(), set(), set())
tf_metabolite_counts = (
    hits.assign(
        met_id=hits["perturbation"].map(lambda p: sorted(pert_sets.get(p, EMPTY)[1]))
    )
    .explode("met_id")
    .dropna(subset=["met_id"])
    .query("met_id not in @CURRENCY_METABOLITES")
    .groupby(["TF", "met_id", "effector_names"], dropna=False)
    .size()
    .reset_index(name="count")
    .assign(metabolite=lambda d: d["met_id"].map(lambda b: bigg_name.get(b, b)))
    .loc[:, ["TF", "effector_names", "met_id", "metabolite", "count"]]
    .sort_values(["count"], ascending=[False])
    .reset_index(drop=True)
)
print(
    f"{len(tf_metabolite_counts)} (TF, metabolite) rows "
    f"across {tf_metabolite_counts['TF'].nunique()} TFs"
)
display(tf_metabolite_counts)
tf_metabolite_counts.to_csv(
    "/workspace/experiments/06222026_tf/06252026_tf_metabolite_counts.csv", index=False
)

In [ ]:
hits.to_csv("06252026_tf_metabolic_pr_top5_hits_per_TF.csv", index=False)

## 6. enveloppe stress

In [ ]:
envelope_pathway = {
    # === FATTY ACID SYNTHESIS (FAS-II): supplies all acyl chains ===
    "fas_ii": [
        "accA",
        "accB",
        "accC",
        "accD",  # acetyl-CoA carboxylase -> malonyl-CoA
        "fabD",  # malonyl-CoA:ACP transacylase
        "fabH",  # 3-oxoacyl-ACP synthase III (initiation)        # HIT
        "fabB",  # 3-oxoacyl-ACP synthase I (elongation, unsat.)
        "fabF",  # 3-oxoacyl-ACP synthase II (elongation)
        "fabG",  # 3-oxoacyl-ACP reductase
        "fabA",  # 3-hydroxyacyl-ACP dehydratase/isomerase
        "fabZ",  # 3-hydroxyacyl-ACP dehydratase
        "fabI",  # enoyl-ACP reductase (cycle closure)            # HIT
    ],
    # === LIPID A (Raetz pathway) ===
    "lipid_a": [
        "lpxA",  # UDP-GlcNAc acyltransferase
        "lpxC",  # UDP-3-O-acyl-GlcNAc deacetylase (1st committed) # HIT
        "lpxD",  # UDP-3-O-acylglucosamine N-acyltransferase       # HIT
        "lpxH",  # UDP-2,3-diacylglucosamine pyrophosphatase
        "lpxB",  # lipid A disaccharide synthase                   # HIT
        "lpxK",  # tetraacyldisaccharide-1-P 4'-kinase -> lipid IVA
        "waaA",  # (kdtA) KDO transferase  -- needs CMP-KDO below
        "lpxL",  # (htrB) lauroyl transferase
        "lpxM",  # (msbB) myristoyl transferase -> mature lipid A
    ],
    # === KDO PRECURSOR BRANCH (feeds waaA) ===
    "kdo_branch": [
        "kdsD",  # D-arabinose-5-P isomerase (gutQ paralog)        # HIT
        "kdsA",  # KDO-8-P synthase
        "kdsC",  # KDO-8-P phosphatase
        "kdsB",  # CMP-KDO synthetase -> CMP-KDO
    ],
    # === CORE + O-ANTIGEN (LPS assembly, post lipid A) ===
    "core_oag": [
        "waaC",
        "waaF",  # heptosyltransferases (inner core)
        "waaG",
        "waaO",
        "waaB",  # gluc/gal core extension
        "waaQ",
        "waaP",
        "waaY",  # core heptose modification/phosphorylation
        "wzzB",
        "wzx",
        "wzy",  # O-antigen chain length / flippase / polymerase
        "waaL",  # O-antigen ligase
    ],
    # === GLYCEROPHOSPHOLIPID BRANCH (shares acyl-ACP pool) ===
    "glycerophospholipid": [
        "plsB",  # glycerol-3-P acyltransferase
        "plsC",  # 1-acyl-G3P acyltransferase -> phosphatidic acid  # HIT
        "cdsA",  # CDP-diacylglycerol synthase
        "pgsA",  # PGP synthase
        "pgpA",
        "pgpB",
        "pgpC",  # PGP phosphatases -> phosphatidylglycerol
        "pssA",  # phosphatidylserine synthase
        "psd",  # PS decarboxylase -> phosphatidylethanolamine
        "clsA",
        "clsB",
        "clsC",  # cardiolipin synthases
    ],
    # === ISOPRENOID / UNDECAPRENYL-P (O-antigen + PG carrier lipid) ===
    "isoprenoid_und_p": [
        "dxs",  # MEP pathway entry
        "dxr",  # (ispC)
        "ispD",
        "ispE",
        "ispF",
        "ispG",
        "ispH",  # -> IPP / DMAPP
        "ispA",  # FPP synthase                                    # HIT
        "uppS",  # undecaprenyl-PP synthase
        "uppP",  # (bacA) undecaprenyl-PP phosphatase -> und-P
    ],
    # === LPS EXPORT (Lpt trans-envelope bridge) ===
    "lpt_export": [
        "msbA",  # IM flippase (lipid A-core to periplasmic leaflet)
        "lptB",  # ABC ATPase
        "lptF",  # IM permease
        "lptG",  # IM permease                                     # HIT
        "lptC",  # IM-anchored bridge start
        "lptA",  # periplasmic bridge
        "lptE",  # OM plug
        "lptD",  # OM translocon (terminal LPS insertion)          # HIT
    ],
}

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

TF_NAME = "gadw"  # which TF column to plot

# --- pull the TF column, case-insensitive ---
# z.Z_emp rows are aligned to `mom.perts` (sorted-unique val perturbations),
# NOT to the per-cell `perts` Series — zipping against `perts` mispairs every
# gene with an unrelated row's score. Key the lookup on mom.perts.
row_perts = np.asarray(mom.perts).astype(str)
assert len(row_perts) == z.Z_emp.shape[0], "row axis mismatch with z.Z_emp"
tf_genes = np.asarray(tf_genes)
tf_lc = np.char.lower(tf_genes.astype(str))
tf_col = np.where(tf_lc == TF_NAME)[0][0]
z_tf = {p.lower(): v for p, v in zip(row_perts, z.Z_emp[:, tf_col])}

HITS = {
    g.lower()
    for g in {
        "fabH",
        "fabI",
        "lpxC",
        "lpxD",
        "lpxB",
        "kdsD",
        "plsC",
        "ispA",
        "lptG",
        "lptD",
    }
}

# --- flatten the pathway dict (genes already lowercase) ---
rows, colors, ticklabels, group_spans = [], [], [], []
cmap = plt.get_cmap("tab10")
y = 0
for gi, (pathway, genes) in enumerate(envelope_pathway.items()):
    present = [g.lower() for g in genes if g.lower() in z_tf]
    start = y
    for g in present:
        rows.append((y, z_tf[g]))
        colors.append(cmap(gi % 10))
        ticklabels.append(f"{g}{' *' if g in HITS else ''}")
        y += 1
    if present:
        group_spans.append((pathway, start, y - 1, cmap(gi % 10)))
    y += 1

fig, ax = plt.subplots(figsize=(7, 0.28 * y + 1))
ys, val = [r[0] for r in rows], [r[1] for r in rows]
ax.barh(ys, val, color=colors, edgecolor="none")
for t in (-10, 10):
    ax.axvline(t, color="k", ls="--", lw=0.8, alpha=0.5)
ax.axvline(0, color="k", lw=0.6)
ax.set_yticks(ys)
ax.set_yticklabels(ticklabels, fontsize=8)
ax.invert_yaxis()
ax.set_xlabel("signed $Z_{emp}$")
ax.set_title(f"{TF_NAME} regulon response across envelope pathway")
for pathway, lo, hi, c in group_spans:
    ax.text(
        1.02,
        (lo + hi) / 2,
        pathway,
        transform=ax.get_yaxis_transform(),
        va="center",
        ha="left",
        fontsize=8,
        color=c,
        fontweight="bold",
    )
plt.tight_layout()
plt.show()